# Integration benchmark subsetting

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os

In [ ]:
os.chdir('SET TO YOUR WORKING DIRECTORY')

In [ ]:
adata = sc.read_h5ad('hca-gut-atlas-tutorial/data/hgca_full_object_published.h5ad')

In [ ]:
pd.crosstab(adata.obs['dataset_id'], adata.obs['closest_GCA_celltype'])

closest_GCA_celltype,B Memory,B Naive,CD4 T,CD8 T,Colonocytes,DC cDC2,Endothelial,Enterocytes,Enteroendocrine Cells (EEC),Epithelial Stem Cells (LGR5+),...,Macrophages M1,NK Cells,Pericytes,Plasma Cells,Secretory Goblet Cells,Secretory Goblet cells Mature,Secretory Paneth Cells,Secretory Tuft Cells,T Unconventional,Transiently Amplifying Cells (TA)
dataset_id,,,,,,,,,,,,,,,,,,,,,
ArendsHelmsley,220,175,804,749,6,355,135,6189,175,654,...,483,42,95,460,1545,3677,6,509,5,4
BasuGCARNA,3668,1279,5118,5508,14219,536,1185,36269,554,3822,...,1061,222,162,10971,7028,3753,85,462,83,2772
Burclaff2022,0,0,0,0,452,0,0,4032,145,3339,...,0,0,0,0,1093,273,29,616,0,410
Dominguez2022,85,25,346,719,0,14,0,0,0,0,...,44,17,0,130,0,0,0,0,0,0
DominguezUnpub,71,11,10311,20378,0,105,0,0,0,0,...,372,293,0,2089,0,0,0,0,8,0
DominguezUnpub2,1288,503,23997,93439,0,1104,0,0,0,0,...,3090,4949,0,5871,0,0,0,0,138,0
Egozi2023,4,0,68,16,0,0,0,0,0,0,...,0,13,0,0,0,0,0,0,1,0
Elmentaite2020,3361,2014,8296,5979,6880,935,2408,8140,71,4428,...,648,324,513,3004,1500,393,24,53,534,1215
He2020,1962,1784,2187,1353,193,564,101,1763,23,68,...,106,18,134,424,140,19,0,95,2,108


In [ ]:
MIN_CELLS, FRACTION, SEED = 150, 0.40, 0
DATASET, SAMPLE = "dataset_id", "sample_id"

In [ ]:
obs, rng = adata.obs, np.random.default_rng(SEED)

In [ ]:
sizes = obs[SAMPLE].value_counts()
target = round(FRACTION * obs[SAMPLE].nunique())

In [ ]:
# eligible samples (>= MIN_CELLS cells), grouped per dataset, in random order
pairs = obs[[DATASET, SAMPLE]].drop_duplicates()
pairs = pairs[pairs[SAMPLE].map(sizes) >= MIN_CELLS]
pools = {ds: list(rng.permutation(g[SAMPLE].to_numpy())) for ds, g in pairs.groupby(DATASET, observed=True)}
order = list(rng.permutation(list(pools)))

In [ ]:
subset_ids = []
while len(subset_ids) < target and any(pools.values()):
    for ds in order:                              # one sample per dataset per round
        if pools[ds]:
            subset_ids.append(pools[ds].pop())
            if len(subset_ids) >= target:
                break

In [ ]:
sub = adata[obs[SAMPLE].isin(subset_ids)].copy()
for c in sub.obs.select_dtypes("category"):       # drop categories that lost all cells
    sub.obs[c] = sub.obs[c].cat.remove_unused_categories()

In [ ]:
print(f"{len(subset_ids)} / {obs[SAMPLE].nunique()} samples, "
      f"{sub.n_obs:,} / {adata.n_obs:,} cells ({100 * sub.n_obs / adata.n_obs:.1f}%), "
      f"{sub.obs[DATASET].nunique()} datasets")

201 / 502 samples, 457,912 / 944,390 cells (48.5%), 26 datasets


In [ ]:
sub

AnnData object with n_obs × n_vars = 457912 × 35574
    obs: 'sample_id', 'author_cell_type', 'doublet_score', 'annotation_cluster', 'predicted_doublet', 'leiden_lineage_l1', 'leiden_lineage_l2', 'qc_lineage', 'n_counts', 'n_genes', 'donor_id', 'dataset_id', 'tissue_ontology_term', 'tissue_ontology_term_id', 'tissue_free_text', 'sample_source', 'sample_collection_method', 'tissue_type', 'sampled_site_condition', 'disease_ontology_term', 'disease_ontology_term_id', 'sample_preservation_method', 'suspension_type', 'is_primary_data', 'age_range', 'development_stage_ontology_term_id', 'radial_tissue_term', 'dissociation_protocol', 'cell_enrichment', 'sample_collection_year', 'library_id', 'library_id_repository', 'library_preparation_batch', 'library_sequencing_run', 'sample_collection_site', 'sample_collection_relative_time_point', 'cell_number_loaded', 'cell_viability_percentage', 'institute', 'author_batch_notes', 'consortia', 'study_pi', 'contact_email', 'batch_condition', 'default_emb

In [ ]:
pd.crosstab(sub.obs['dataset_id'], sub.obs['closest_GCA_celltype'])

closest_GCA_celltype,B Memory,B Naive,CD4 T,CD8 T,Colonocytes,DC cDC2,Endothelial,Enterocytes,Enteroendocrine Cells (EEC),Epithelial Stem Cells (LGR5+),...,Macrophages M1,NK Cells,Pericytes,Plasma Cells,Secretory Goblet Cells,Secretory Goblet cells Mature,Secretory Paneth Cells,Secretory Tuft Cells,T Unconventional,Transiently Amplifying Cells (TA)
dataset_id,,,,,,,,,,,,,,,,,,,,,
ArendsHelmsley,220,175,804,749,6,355,135,6189,175,654,...,483,42,95,460,1545,3677,6,509,5,4
BasuGCARNA,931,549,1979,2597,5517,238,491,16609,257,1824,...,485,114,70,5294,2569,2016,40,220,33,810
Burclaff2022,0,0,0,0,445,0,0,1497,97,2513,...,0,0,0,0,1015,140,8,508,0,365
Dominguez2022,44,23,308,630,0,12,0,0,0,0,...,43,16,0,126,0,0,0,0,0,0
DominguezUnpub,71,11,10311,20378,0,105,0,0,0,0,...,372,293,0,2089,0,0,0,0,8,0
DominguezUnpub2,130,103,6398,46684,0,89,0,0,0,0,...,472,660,0,1945,0,0,0,0,36,0
Elmentaite2020,429,149,729,493,1110,79,439,2269,23,546,...,24,48,34,14,207,67,5,8,5,144
He2020,1962,1784,2187,1353,193,564,101,1763,23,68,...,106,18,134,424,140,19,0,95,2,108
Huang2019,2862,6132,3760,3296,23,20,0,5,0,2,...,18,167,0,64,7,1,0,104,766,0


In [ ]:
sub.write_h5ad("data/hgca_benchmark_subset.h5ad")